#Environment Setup & Configurations

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

API_KEY = "UR API key"
HEADERS = {"X-API-Key": API_KEY, "accept": "application/json"}
SENSOR_ID = 7041505

DATE_FROM = "2023-09-01T00:00:00Z"
DATE_TO = "2026-08-31T23:59:59Z"

print("Libraries imported and configuration set successfully.")

Libraries imported and configuration set successfully.


#Ground Truth Ingestion & Unit Conversion Routine

In [ ]:

def handle_unit_conversions(value: float, parameter: str, temp_c: float = 25.0, pressure_hpa: float = 1013.25) -> float:
    """
    Physical unit conversion routine (µg/m³ to ppb) using the Ideal Gas Law.
    Applicable for gaseous pollutants (NO2, SO2, O3, CO).
    Particulate matter (PM2.5, PM10) remains standardized in µg/m³.
    """
    molar_masses = {
        "no2": 46.0055,
        "o3": 47.9982,
        "so2": 64.066,
        "co": 28.0101
    }
    param = parameter.lower().replace(".", "")
    if param in molar_masses:
        R = 8.31446
        temp_k = temp_c + 273.15
        pressure_pa = pressure_hpa * 100.0
        v_molar = (R * temp_k) / pressure_pa
        return value * (v_molar / molar_masses[param])

    return value

url = f"https://api.openaq.org/v3/sensors/{SENSOR_ID}/measurements"
records = []
page = 1
limit = 1000

print(f"Fetching ground measurements from OpenAQ v3 ({DATE_FROM} to {DATE_TO})...")

while True:
    params = {
        "datetime_from": DATE_FROM,
        "datetime_to": DATE_TO,
        "limit": limit,
        "page": page
    }
    response = requests.get(url, headers=HEADERS, params=params)
    if response.status_code != 200:
        print(f"Fetch completed or stopped at page {page} with status code: {response.status_code}")
        break

    data = response.json().get("results", [])
    if not data:
        break

    for item in data:
        raw_val = item.get("value")
        std_val = handle_unit_conversions(raw_val, parameter="pm25")
        records.append({
            "timestamp_utc": pd.to_datetime(item.get("period", {}).get("datetimeFrom", {}).get("utc")),
            "station_pm25": std_val,
            "parameter": "pm25",
            "unit": "ug/m3"
        })

    if len(data) < limit:
        break
    page += 1

df_ground = pd.DataFrame(records)
df_ground = df_ground.drop_duplicates(subset=["timestamp_utc"]).sort_values("timestamp_utc").reset_index(drop=True)

print(f"Ground Truth Records Retrieved: {len(df_ground):,}")
print(f"Temporal Coverage: {df_ground['timestamp_utc'].min()} -> {df_ground['timestamp_utc'].max()}")

# Export ground truth dataset
df_ground.to_csv("openaq_cairo_ground_truth_3years.csv", index=False, encoding="utf-8-sig")

Fetching ground measurements from OpenAQ v3 (2023-09-01T00:00:00Z to 2026-08-31T23:59:59Z)...
Fetch completed or stopped at page 3 with status code: 500
Ground Truth Records Retrieved: 2,000
Temporal Coverage: 2023-09-25 11:00:00+00:00 -> 2023-12-18 04:00:00+00:00


#Spatial & Temporal Alignment (Downtown Cairo)

In [ ]:

meteo_filename = "Merged_AQ_Weather_Data_Meteo.csv"
if not os.path.exists(meteo_filename):
    raise FileNotFoundError(f"Missing input file: {meteo_filename}")

df_meteo = pd.read_csv(meteo_filename)

df_downtown = df_meteo[df_meteo["Location"].astype(str).str.contains("Downtown", case=False, na=False)].copy()

df_ground["hour_str"] = pd.to_datetime(df_ground["timestamp_utc"], utc=True).dt.strftime("%Y-%m-%d %H:00")
df_downtown["hour_str"] = pd.to_datetime(df_downtown["date"], utc=True).dt.strftime("%Y-%m-%d %H:00")

ground_hourly = df_ground.groupby("hour_str")["station_pm25"].mean().reset_index()
meteo_hourly = df_downtown.groupby("hour_str")["pm2_5"].mean().reset_index().rename(columns={"pm2_5": "model_pm25"})

df_comparison = pd.merge(ground_hourly, meteo_hourly, on="hour_str", how="inner").dropna()

df_comparison.to_csv("cairo_pm25_validation_comparison_3years.csv", index=False, encoding="utf-8-sig")
print(f"Successfully matched {len(df_comparison):,} overlapping hours for Downtown Cairo.")

Successfully matched 2,000 overlapping hours for Downtown Cairo.


#Statistical Validation & Metric Evaluation

In [ ]:

if len(df_comparison) > 0:
    y_true = df_comparison["station_pm25"].values
    y_pred = df_comparison["model_pm25"].values

    mae = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(np.mean((y_pred - y_true)**2))
    bias = np.mean(y_pred - y_true)
    nmb = (np.sum(y_pred - y_true) / np.sum(y_true)) * 100.0

    denom = np.sqrt(np.sum((y_true - np.mean(y_true))**2) * np.sum((y_pred - np.mean(y_pred))**2))
    pearson_r = np.sum((y_true - np.mean(y_true)) * (y_pred - np.mean(y_pred))) / denom if denom != 0 else 0.0

    print("=" * 60)
    print("      DOWNTOWN CAIRO MODEL VALIDATION REPORT (3 YEARS)      ")
    print("=" * 60)
    print(f"• Matched Hourly Observations (N) : {len(df_comparison):,}")
    print(f"• Ground Station Mean (True PM2.5): {np.mean(y_true):.2f} µg/m³")
    print(f"• Open-Meteo Mean (Model PM2.5)   : {np.mean(y_pred):.2f} µg/m³")
    print(f"• Mean Absolute Error (MAE)       : {mae:.2f} µg/m³")
    print(f"• Root Mean Squared Error (RMSE)  : {rmse:.2f} µg/m³")
    print(f"• Mean Bias                       : {bias:.2f} µg/m³")
    print(f"• Normalized Mean Bias (NMB)      : {nmb:.2f}%")
    print(f"• Pearson Correlation (r)         : {pearson_r:.3f}")
    print("=" * 60)

    files.download("openaq_cairo_ground_truth_3years.csv")
    files.download("cairo_pm25_validation_comparison_3years.csv")

      DOWNTOWN CAIRO MODEL VALIDATION REPORT (3 YEARS)      
• Matched Hourly Observations (N) : 2,000
• Ground Station Mean (True PM2.5): 57.74 µg/m³
• Open-Meteo Mean (Model PM2.5)   : 26.46 µg/m³
• Mean Absolute Error (MAE)       : 32.17 µg/m³
• Root Mean Squared Error (RMSE)  : 50.09 µg/m³
• Mean Bias                       : -31.28 µg/m³
• Normalized Mean Bias (NMB)      : -54.17%
• Pearson Correlation (r)         : 0.300


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>